# PyTorchのテンソル（tensor）の扱い方②：`max` 関数／メソッド

このノートブックでは、PyTorch の **`max` 関数／メソッド** を扱います。
強化学習、とくに DQN のコードを読むときに頻出で、
`max(1)[0]` や `max(1)[1]` のような書き方がよく出てきます。

一般的な DQN 実装では、例えば次のような式がよく登場します。

- `q_network(next_states).max(1)[0].detach()`
- `q_network(state).max(1)[1].view(1, 1)`

これらを **直感的に理解できるようにする** のが、このノートブックの目的です。

## このノートブックで学ぶこと（目次）

1. **`torch.max` / `Tensor.max` の基本**  
   どんな引数パターンがあって、何を返すのかを整理します。
2. **`max(dim)` が返す「値」と「インデックス」**  
   `max(1)[0]` と `max(1)[1]` の意味を、簡単なテンソルで確認します。
3. **Q値テーブルに対する `max`：最大Q値と、そのときの行動**  
   「行ごとに最大値」と「その列番号」を取り出す例を作ります。
4. **DQN のコードでの `max` の使われ方**  
   典型的な DQN 風のコードを、小さな例に置き換えて眺めます。

## 1. `torch.max` / `Tensor.max` の基本イメージ

PyTorch の `max` には、主に次の 3 パターンがあります。

1. **テンソル全体の最大値を 1 つ返す**  
   - 例: `torch.max(x)` / `x.max()`
   - 返り値: スカラー（テンソルだが中身は 1 つの値）

2. **ある次元 `dim` に沿って最大値を返す（値とインデックスのペア）**  
   - 例: `torch.max(x, dim=1)` / `x.max(dim=1)`
   - 返り値: `namedtuple(values, indices)` もしくは `(values, indices)`
     - `values`: 各行（または各列）ごとの最大値
     - `indices`: その最大値が「どの位置（インデックス）」にあったか

3. **ブロードキャスト可能な 2 つのテンソルの要素ごとの最大値**  
   - 例: `torch.max(x, y)`  
   - これは本ノートでは深追いしません（DQN の文脈ではあまり出てこないため）。

DQN でよく出てくるのは **2. の「`dim` に沿った最大値」** です。

- `max(1)` と書いてあるときは、たいてい **「行ごと（dim=1）に、列方向で最大値を取る」** 意味です。
- その返り値は **「(値テンソル, インデックステンソル) のペア」** なので、
  - `max(1)[0]` が「値」
  - `max(1)[1]` が「インデックス（位置）」
 という役割になります。

In [5]:
import torch

# 1. テンソル全体の最大値
x = torch.tensor([1.0, -3.0, 5.0, 2.0])
print("x =", x)

print("\n全体の最大値 (torch.max):", torch.max(x))
print("全体の最大値 (x.max):", x.max())

# 2. 2次元テンソルに対して dim を指定して max を取る
q = torch.tensor([
    [1.0, 2.0, 3.0],
    [0.5, 10.0, -1.0]
])
print("\nq =")
print(q)

# dim=1: 行ごとに「列方向で」最大値を取る
values, indices = q.max(dim=1)
print("\nvalues (各行の最大値) =", values)
print("indices (その列インデックス) =", indices)

# dim=0: 列ごとに「行方向で」最大値を取る
values0, indices0 = q.max(dim=0)
print("\nvalues0 (各列の最大値) =", values0)
print("indices0 (その行インデックス) =", indices0)

x = tensor([ 1., -3.,  5.,  2.])

全体の最大値 (torch.max): tensor(5.)
全体の最大値 (x.max): tensor(5.)

q =
tensor([[ 1.0000,  2.0000,  3.0000],
        [ 0.5000, 10.0000, -1.0000]])

values (各行の最大値) = tensor([ 3., 10.])
indices (その列インデックス) = tensor([2, 1])

values0 (各列の最大値) = tensor([ 1., 10.,  3.])
indices0 (その行インデックス) = tensor([0, 1, 0])


## 2. `max(dim)` の返り値と `max(1)[0]`, `max(1)[1]`

先ほどの `q.max(dim=1)` のように、`dim` を指定して `max` を呼び出すと、
**「(values, indices) の 2 つ」** が返ってきます。

- **`values`**: 各行（または各列）の最大値そのもの
- **`indices`**: その最大値が「どの位置（インデックス）」にあったか

典型的な書き方は次の 2 パターンです。

- 代入して受け取る:  
  `values, indices = q.max(dim=1)`
- インデックスで取り出す:  
  `q.max(1)[0]` → 値（values）だけ  
  `q.max(1)[1]` → インデックス（indices）だけ

DQN のコードで出てくる

- `...max(1)[0]` は「最大 **Q値** を取り出す」
- `...max(1)[1]` は「最大 Q値を与える **行動インデックス** を取り出す」

という意味になっています。

このあと、小さな Q値テーブルを使って、これを確認してみます。

In [6]:
import torch

# Q値テーブルのイメージ: 行が状態、列が行動（例: 左, 右, 何もしない）
q_values = torch.tensor([
    [1.0, 2.0, 3.0],   # 状態s0でのQ値
    [0.5, 10.0, -1.0], # 状態s1でのQ値
    [5.0, 4.0, 6.0]    # 状態s2でのQ値
])

print("q_values =")
print(q_values)

# 行ごと（dim=1）に、「列方向」で最大値を取る
values, indices = q_values.max(dim=1)
print("\nvalues (各状態での最大Q値) =", values)
print("indices (そのときの行動インデックス) =", indices)

# 同じことを、[0]/[1] で書くと…
print("\nq_values.max(1)[0] =", q_values.max(1)[0])  # 最大Q値
print("q_values.max(1)[1] =", q_values.max(1)[1])  # argmax（行動のindex）

q_values =
tensor([[ 1.0000,  2.0000,  3.0000],
        [ 0.5000, 10.0000, -1.0000],
        [ 5.0000,  4.0000,  6.0000]])

values (各状態での最大Q値) = tensor([ 3., 10.,  6.])
indices (そのときの行動インデックス) = tensor([2, 1, 2])

q_values.max(1)[0] = tensor([ 3., 10.,  6.])
q_values.max(1)[1] = tensor([2, 1, 2])


## 3. Q学習の式と $\max_a Q(s_{t+1}, a)$

Q学習（DQN）では、次のような更新式が出てきます（細かいところはここでは雰囲気だけ）。

$$
Q(s_t, a_t) \leftarrow r_{t+1} + \gamma \max_a Q(s_{t+1}, a)
$$

右辺の

- $\max_a Q(s_{t+1}, a)$ が、
- PyTorch のコードでは **「次状態 $s_{t+1}$ に対する Q値ベクトルの、最大値」**

として実装されます。

典型的な DQN 風のコードでは、

- `q_network(next_states)` が、
  - 形状 `[batch_size_non_final, num_actions]` の Q値テーブル（各行が状態、各列が行動）を出力し、
- そこに `max(1)[0]` をかけて、**各行（各次状態）ごとの最大Q値** を取り出します。

このイメージを、小さな例で確認してみます。

In [7]:
import torch

# next_q_values: 「次状態」に対する Q値テーブルのイメージ
#   行: 各サンプルの次状態 s_{t+1}
#   列: 行動（例: 左=0, 右=1）
next_q_values = torch.tensor([
    [1.0, 2.0],   # サンプル0の次状態でのQ値
    [5.0, 3.0],   # サンプル1
    [0.0, -1.0]   # サンプル2
])

print("next_q_values =")
print(next_q_values)

# 各次状態ごとに、max_a Q(s_{t+1}, a) を取り出す
max_next_q_values, argmax_actions = next_q_values.max(dim=1)

print("\nmax_next_q_values (max_a Q(s_{t+1}, a)) =", max_next_q_values)
print("argmax_actions (そのときの行動index) =", argmax_actions)

# DQNコードの書き方に合わせると…
print("\nnext_q_values.max(1)[0] =", next_q_values.max(1)[0])  # 最大Q値
print("next_q_values.max(1)[1] =", next_q_values.max(1)[1])  # 行動index

next_q_values =
tensor([[ 1.,  2.],
        [ 5.,  3.],
        [ 0., -1.]])

max_next_q_values (max_a Q(s_{t+1}, a)) = tensor([2., 5., 0.])
argmax_actions (そのときの行動index) = tensor([1, 0, 0])

next_q_values.max(1)[0] = tensor([2., 5., 0.])
next_q_values.max(1)[1] = tensor([1, 0, 0])


## 4. DQN 風のコードに対応づけてみる

一般的な DQN 風の実装では、代表的に次の 2 つの `max` が出てきます。

1. **学習時（教師信号の計算）での `max`**  
   ```python
   next_state_values[non_final_mask] = q_network(
       non_final_next_states
   ).max(1)[0].detach()
   ```
   - `q_network(non_final_next_states)` は、形状 `[N_non_final, num_actions]` の Q値テーブル
   - `.max(1)[0]` は、「各行について、列方向（行動方向）の最大Q値」を取り出す
   - これは数式でいう `max_a Q(s_{t+1}, a)` に対応

2. **行動選択（ε-greedy）での `max`**  
   ```python
   action = q_network(state).max(1)[1].view(1, 1)
   ```
   - `q_network(state)` は、形状 `[1, num_actions]` の Q値ベクトル
   - `.max(1)[1]` は、その行の中で「最大Q値を与える列インデックス（行動番号）」
   - `.view(1, 1)` で `[1]` → `[[1]]` の形に整えている

次のセルでは、この 2 つに対応する **極小版のコード** を書いて、形状や中身をプリントして確かめます。

In [8]:
import torch

# --- 4.1 学習時の max(1)[0] の極小例 ---

# 「non_final_next_states に対する Q値」をイメージしたテンソル
# 行: 各サンプルの次状態, 列: 行動0, 行動1
q_next = torch.tensor([
    [1.0, 2.0],   # サンプル0
    [5.0, 3.0],   # サンプル1
])

print("q_next =")
print(q_next)
print("q_next.shape =", q_next.shape)

# DQNコード: self.model(non_final_next_states).max(1)[0]
max_q_next = q_next.max(1)[0]
print("\nq_next.max(1)[0] (各次状態の最大Q値) =", max_q_next)
print("max_q_next.shape =", max_q_next.shape)

# --- 4.2 行動選択時の max(1)[1] の極小例 ---

# ある状態 s に対する Q値ベクトル（model(state) のイメージ）
q_for_state = torch.tensor([[1.0, 4.0]])  # 形状 [1, 2]
print("\nq_for_state =")
print(q_for_state)
print("q_for_state.shape =", q_for_state.shape)

# DQNコード: self.model(state).max(1)[1].view(1, 1)
best_action_index = q_for_state.max(1)[1].view(1, 1)
print("\nq_for_state.max(1)[1] =", q_for_state.max(1)[1])
print("best_action_index = q_for_state.max(1)[1].view(1, 1) =")
print(best_action_index)
print("best_action_index.shape =", best_action_index.shape)

q_next =
tensor([[1., 2.],
        [5., 3.]])
q_next.shape = torch.Size([2, 2])

q_next.max(1)[0] (各次状態の最大Q値) = tensor([2., 5.])
max_q_next.shape = torch.Size([2])

q_for_state =
tensor([[1., 4.]])
q_for_state.shape = torch.Size([1, 2])

q_for_state.max(1)[1] = tensor([1])
best_action_index = q_for_state.max(1)[1].view(1, 1) =
tensor([[1]])
best_action_index.shape = torch.Size([1, 1])
